# Momentum

## Learning Objectives
1. Implement SGD, momentum, and Nesterov momentum on the Rosenbrock function using NumPy.
2. Compare optimisers on a neural network in PyTorch with OOM-safe training.
3. Demonstrate momentum warm-up for stable large-batch training.
4. Show how high momentum causes gradient explosion and how gradient clipping fixes it.


In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1: SGD / Momentum / Nesterov on Rosenbrock (NumPy)


In [ ]:
def rosenbrock(x: np.ndarray, a: float = 1.0, b: float = 100.0) -> float:
    """Rosenbrock banana function: f = (a - x0)^2 + b*(x1 - x0^2)^2."""
    return (a - x[0]) ** 2 + b * (x[1] - x[0] ** 2) ** 2


def rosenbrock_grad(x: np.ndarray, a: float = 1.0, b: float = 100.0) -> np.ndarray:
    """Gradient of Rosenbrock."""
    gx0 = -2 * (a - x[0]) - 4 * b * x[0] * (x[1] - x[0] ** 2)
    gx1 = 2 * b * (x[1] - x[0] ** 2)
    return np.array([gx0, gx1])


def run_optimiser(name: str, lr=0.001, momentum=0.9, n_steps=2000):
    """Run one of {sgd, momentum, nesterov} on Rosenbrock."""
    x = np.array([-1.5, 1.0])
    v = np.zeros_like(x)
    path = [x.copy()]

    for _ in range(n_steps):
        if name == "sgd":
            g = rosenbrock_grad(x)
            x = x - lr * g
        elif name == "momentum":
            g = rosenbrock_grad(x)
            v = momentum * v - lr * g
            x = x + v
        elif name == "nesterov":
            x_ahead = x + momentum * v
            g = rosenbrock_grad(x_ahead)
            v = momentum * v - lr * g
            x = x + v
        path.append(x.copy())

    return np.array(path)


paths = {
    "sgd":      run_optimiser("sgd",      lr=0.0005),
    "momentum": run_optimiser("momentum", lr=0.0005, momentum=0.9),
    "nesterov": run_optimiser("nesterov", lr=0.0005, momentum=0.9),
}

for name, path in paths.items():
    final_x = path[-1]
    print(f"{name:>12}: final x={final_x}, f={rosenbrock(final_x):.6f}, "
          f"best step={np.argmin([rosenbrock(p) for p in path]):4d}")


## Level 2: Optimiser Comparison on Neural Network (PyTorch + OOM Handling)


In [ ]:
class RegressionMLP(nn.Module):
    """MLP for regression benchmarking."""
    def __init__(self, in_dim=20, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x): return self.net(x)


X_reg = torch.randn(2000, 20)
y_reg = (X_reg[:, 0] * X_reg[:, 1] + torch.sin(X_reg[:, 2])).unsqueeze(1)
reg_loader = DataLoader(TensorDataset(X_reg[:1600], y_reg[:1600]),
                        batch_size=64, shuffle=True)
X_val_r, y_val_r = X_reg[1600:].to(device), y_reg[1600:].to(device)


def train_opt(opt_name: str, n_epochs=50, momentum=0.9, lr=5e-3):
    """Train with named optimiser, return per-epoch val losses."""
    m = RegressionMLP().to(device)
    if opt_name == "sgd":
        opt = torch.optim.SGD(m.parameters(), lr=lr)
    elif opt_name == "momentum":
        opt = torch.optim.SGD(m.parameters(), lr=lr, momentum=momentum)
    elif opt_name == "nesterov":
        opt = torch.optim.SGD(m.parameters(), lr=lr, momentum=momentum, nesterov=True)
    else:
        opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    crit = nn.MSELoss()
    val_losses = []
    for _ in range(n_epochs):
        m.train()
        for xb, yb in reg_loader:
            opt.zero_grad()
            try:
                loss = crit(m(xb.to(device)), yb.to(device))
            except RuntimeError as exc:
                if "out of memory" in str(exc).lower():
                    print(f"OOM in {opt_name}: reduce batch_size")
                    torch.cuda.empty_cache(); continue
                raise
            loss.backward(); opt.step()
        m.eval()
        with torch.no_grad():
            val_losses.append(crit(m(X_val_r), y_val_r).item())
    return val_losses


results_opt = {}
for name in ["sgd", "momentum", "nesterov", "adam"]:
    results_opt[name] = train_opt(name)
    print(f"{name:>12}: final val MSE = {results_opt[name][-1]:.4f}")


## Real-World Example 1: Momentum Warm-Up Schedule


In [ ]:
class WarmupMomentumSGD:
    """SGD with linear momentum warm-up over warmup_steps."""
    def __init__(self, params, lr=0.01, target_momentum=0.9, warmup_steps=100):
        self.optimizer = torch.optim.SGD(params, lr=lr, momentum=0.0)
        self.target_momentum = target_momentum
        self.warmup_steps = warmup_steps
        self._step = 0

    def step(self):
        self._step += 1
        ratio = min(self._step / self.warmup_steps, 1.0)
        new_m = ratio * self.target_momentum
        for pg in self.optimizer.param_groups:
            pg['momentum'] = new_m
        self.optimizer.step()

    def zero_grad(self):
        self.optimizer.zero_grad()


m_warm = RegressionMLP().to(device)
warm_opt = WarmupMomentumSGD(m_warm.parameters(), lr=5e-3,
                              target_momentum=0.9, warmup_steps=200)
m_cold = RegressionMLP().to(device)
cold_opt = torch.optim.SGD(m_cold.parameters(), lr=5e-3, momentum=0.9)
crit_rw1 = nn.MSELoss()

warm_losses, cold_losses = [], []
for epoch in range(80):
    for xb, yb in reg_loader:
        xb, yb = xb.to(device), yb.to(device)
        warm_opt.zero_grad()
        crit_rw1(m_warm(xb), yb).backward()
        warm_opt.step()
        cold_opt.zero_grad()
        crit_rw1(m_cold(xb), yb).backward()
        cold_opt.step()
    with torch.no_grad():
        warm_losses.append(crit_rw1(m_warm(X_val_r), y_val_r).item())
        cold_losses.append(crit_rw1(m_cold(X_val_r), y_val_r).item())

print(f"Warmup SGD final val MSE : {warm_losses[-1]:.4f}")
print(f"Cold-start SGD final MSE : {cold_losses[-1]:.4f}")
print(f"Improvement: {cold_losses[-1] - warm_losses[-1]:+.4f}")


## Real-World Example 2: Gradient Explosion with High Momentum + Clipping Fix


In [ ]:
def train_with_clipping(clip_value=None, momentum=0.99, n_epochs=30):
    """Train on regression; return loss history and whether it diverged."""
    m = nn.Sequential(
        nn.Linear(20, 64), nn.Tanh(),
        nn.Linear(64, 64), nn.Tanh(),
        nn.Linear(64, 1),
    ).to(device)
    opt = torch.optim.SGD(m.parameters(), lr=0.02, momentum=momentum)
    crit = nn.MSELoss()
    losses = []
    diverged = False
    for _ in range(n_epochs):
        epoch_loss = 0.0
        for xb, yb in reg_loader:
            opt.zero_grad()
            loss = crit(m(xb.to(device)), yb.to(device))
            loss.backward()
            if clip_value is not None:
                nn.utils.clip_grad_norm_(m.parameters(), clip_value)
            opt.step()
            epoch_loss += loss.item()
        avg = epoch_loss / len(reg_loader)
        losses.append(avg)
        if np.isnan(avg) or avg > 1e6:
            diverged = True
            break
    return losses, diverged


losses_no_clip, diverged1 = train_with_clipping(clip_value=None,  momentum=0.99)
losses_clipped, diverged2 = train_with_clipping(clip_value=1.0,   momentum=0.99)

print(f"High momentum, NO clipping: diverged={diverged1}, "
      f"final loss={losses_no_clip[-1]:.4f}")
print(f"High momentum, clip=1.0   : diverged={diverged2}, "
      f"final loss={losses_clipped[-1]:.4f}")
print("Gradient clipping prevents divergence when momentum is too high.")


## Real-World Example 3: Convergence Speed Comparison Plot


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = {"sgd": "gray", "momentum": "steelblue", "nesterov": "navy", "adam": "coral"}
for name, losses in results_opt.items():
    axes[0].plot(losses, label=name.capitalize(), color=colors[name], linewidth=2)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Val MSE")
axes[0].set_title("Optimiser Convergence Comparison")
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, min(5, max(results_opt["sgd"][:20]) * 1.1))

axes[1].plot(warm_losses, label="Momentum + Warmup", color="steelblue", linewidth=2)
axes[1].plot(cold_losses, label="Momentum No Warmup", color="coral",
             linestyle="--", linewidth=2)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Val MSE")
axes[1].set_title("Momentum Warm-Up Effect")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/momentum_comparison.png", dpi=80)
plt.close()
print("Convergence comparison plot saved to /tmp/momentum_comparison.png")
